# 3. Silver: `silver_electronics`

Limpeza dos 3 datasets da bronze e gravação em tabelas **Delta** na base de dados `silver_electronics`
(`hdfs://hdfs-nn:9000/silver/silver_electronics.db/`), seguindo a solução do exercício *bronze_silver_gold*.

**Regras comuns do grupo**
- **Fuso horário: UTC.** A sessão Spark corre em UTC, por isso todas as datas são convertidas a partir de UTC.
- **Período: a Silver guarda todas as datas.** O corte ao período comum do grupo é feito na Gold.

| Tabela | 1 linha = | Origem (bronze) |
|---|---|---|
| `amazon_ratings` | 1 avaliação | `Electronics.csv` |
| `amazon_products` | 1 produto da Amazon com avaliações | `meta_Electronics.json` |
| `prices` | 1 preço visto (produto × retalhista × dia) | `DatafinitiElectronicsProductsPricingData.csv` |
| `prices_amazon_link` | 1 par produto Datafiniti ↔ asin da Amazon | `prices` + `amazon_products` |

A ordem importa: os produtos da Amazon dependem das avaliações, e a ligação depende de `prices` e de `amazon_products`.

In [1]:
# Instalar dependências
import sys
!{sys.executable} -m pip install -q delta-spark==2.4.0

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import *

# warehouse_location aponta para a localização por defeito das bases de dados e tabelas
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Silver electronics") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED") \
    .enableHiveSupport()

# session.timeZone = UTC: fuso horário comum do grupo (datas convertidas sempre a partir de UTC)
# timeParserPolicy = CORRECTED: datas que não batem com o formato ficam NULL em vez de dar erro

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
# Caminhos
BRONZE = "hdfs://hdfs-nn:9000/project/bronze"
SILVER = "hdfs://hdfs-nn:9000/silver/silver_electronics.db"

# Os textos da Amazon e da Datafiniti trazem entidades HTML (&amp; em vez de &, etc.)
def limpa_texto(c):
    c = F.regexp_replace(c, "&amp;", "&")
    c = F.regexp_replace(c, "&quot;", '"')
    c = F.regexp_replace(c, "&#39;", "'")
    c = F.regexp_replace(c, "&gt;", ">")
    c = F.regexp_replace(c, "&lt;", "<")
    c = F.trim(c)
    return F.when(c == "", None).otherwise(c)   # texto vazio passa a NULL

## Base de dados silver

In [4]:
# create silver database
spark.sql(
    f"""
    CREATE DATABASE IF NOT EXISTS silver_electronics LOCATION '{SILVER}/'
    """
)

DataFrame[]

## `amazon_ratings`

Problemas encontrados no profiling e o que se faz:
- O CSV **não tem cabeçalho**: os nomes das colunas são dados no schema e passam para `snake_case`.
- Existem **linhas repetidas** (mesmo asin, cliente, nota e data): são removidas.
- `unixReviewTime` está em **segundos desde 1970** e é sempre 00:00 UTC: passa a `DATE` (`review_date`).
- `overall` vem como `DOUBLE` (5.0), mas só toma os valores 1 a 5: passa a `INT` (`rating`).

In [5]:
# create amazon_ratings table in silver
spark.sql("DROP TABLE IF EXISTS silver_electronics.amazon_ratings")

spark.sql(
    f"""
    CREATE EXTERNAL TABLE silver_electronics.amazon_ratings (
        asin STRING,
        reviewer_id STRING,
        rating INT,
        review_date DATE
    )
    USING DELTA
    LOCATION '{SILVER}/amazon_ratings/'
    """
)

DataFrame[]

In [6]:
# read ratings (CSV sem cabeçalho: schema explícito, também evita ler 21 M de linhas duas vezes com inferSchema)
ratings_df = spark.read.csv(
    f"{BRONZE}/Electronics.csv",
    header=False,
    schema="asin STRING, reviewer_id STRING, overall DOUBLE, unix_review_time LONG",
)

# apply filters and enhancements
ratings_df = ratings_df \
    .drop_duplicates() \
    .withColumn("review_date", F.to_date(F.timestamp_seconds("unix_review_time"))) \
    .withColumn("rating", F.col("overall").cast("INT")) \
    .filter(F.col("asin").isNotNull() & F.col("rating").between(1, 5)) \
    .select("asin", "reviewer_id", "rating", "review_date")

# write to delta table
ratings_df \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER}/amazon_ratings/")

# check the results in the table
spark.table("silver_electronics.amazon_ratings").show(5)
spark.table("silver_electronics.amazon_ratings").count()

+----------+--------------+------+-----------+
|      asin|   reviewer_id|rating|review_date|
+----------+--------------+------+-----------+
|0446576476|A2WED9DICNJLKF|     5| 2005-11-05|
|0511189877| AHKMO8X1OE5T1|     5| 2016-09-26|
|0557348153|A3TUFHHA9A31J0|     5| 2013-04-11|
|073530498X|A1KFBWHT92P7NG|     5| 2016-07-11|
|0972683275|A11CLBU0YO7ARC|     5| 2014-10-23|
+----------+--------------+------+-----------+
only showing top 5 rows



20568091

## `amazon_products`

Problemas encontrados no profiling e o que se faz:
- O JSON tem **19 campos**, a maioria texto livre e listas (descrição, imagens, also_buy...): ficam só os 8 que o quality marcou como *Manter*.
- Há **asins repetidos**: fica um registo por asin.
- Muitos produtos não têm nenhuma avaliação no ficheiro de ratings: ficam só os que têm **avaliações** (`left_semi` join com `amazon_ratings`).
- Textos com **entidades HTML** (`Camera &amp; Photo`): passam a `&`.
- `brand` vazia ou genérica (`Generic`, `Unknown`, `N/A`) passa a NULL. `main_cat` com HTML (`<img ...>`) também.
- `category` é uma lista: fica o caminho completo (`category_path`) e o 2.º nível (`sub_category`, o 1.º é sempre *Electronics*).
- `price` é texto (`"$65.00"`, intervalos, vazio): passa a `DOUBLE` só quando é um preço único.
- `date` é texto (`"January 28, 2014"`): passa a `DATE` (`release_date`).
- `rank` é texto (`">#3,092 in Tools..."`): fica o número (`sales_rank`, `INT`).

In [7]:
# create amazon_products table in silver
spark.sql("DROP TABLE IF EXISTS silver_electronics.amazon_products")

spark.sql(
    f"""
    CREATE EXTERNAL TABLE silver_electronics.amazon_products (
        asin STRING,
        title STRING,
        brand STRING,
        main_cat STRING,
        sub_category STRING,
        category_path STRING,
        price DOUBLE,
        release_date DATE,
        sales_rank INT
    )
    USING DELTA
    LOCATION '{SILVER}/amazon_products/'
    """
)

DataFrame[]

In [8]:
# read meta (11 GB: o schema explícito lê só os 8 campos necessários; rank vem às vezes como lista, STRING guarda o texto)
meta_df = spark.read.json(
    f"{BRONZE}/meta_Electronics.json",
    schema="asin STRING, title STRING, brand STRING, main_cat STRING, category ARRAY<STRING>, "
           "date STRING, price STRING, rank STRING",
)

# só os produtos com avaliações
asins_avaliados = spark.table("silver_electronics.amazon_ratings").select("asin").distinct()

marca = limpa_texto(F.col("brand"))
main_cat = limpa_texto(F.col("main_cat"))

# apply filters and enhancements
products_df = meta_df \
    .filter(F.col("asin").isNotNull()) \
    .drop_duplicates(["asin"]) \
    .join(asins_avaliados, "asin", "left_semi") \
    .withColumn("title", limpa_texto(F.col("title"))) \
    .withColumn("brand", F.when(F.lower(marca).isin("generic", "unknown", "n/a", "na", "none"), None).otherwise(marca)) \
    .withColumn("main_cat", F.when(main_cat.startswith("<"), None).otherwise(main_cat)) \
    .withColumn("sub_category", limpa_texto(F.element_at("category", 2))) \
    .withColumn("category_path", limpa_texto(F.concat_ws(" > ", "category"))) \
    .withColumn("price", F.regexp_replace(F.regexp_extract("price", r"^\$([0-9,]+\.[0-9]{2})$", 1), ",", "").cast("DOUBLE")) \
    .withColumn("release_date", F.to_date(F.trim("date"), "MMMM d, yyyy")) \
    .withColumn("sales_rank", F.regexp_replace(F.regexp_extract("rank", r"#([0-9,]+)", 1), ",", "").cast("INT")) \
    .select("asin", "title", "brand", "main_cat", "sub_category", "category_path", "price", "release_date", "sales_rank")

# write to delta table
products_df \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER}/amazon_products/")

# check the results in the table
spark.table("silver_electronics.amazon_products").show(5, 40)
spark.table("silver_electronics.amazon_products").count()

+----------+----------------------------------------+-----------------------------------+--------------------+---------------------------+----------------------------------------+-----+------------+----------+
|      asin|                                   title|                              brand|            main_cat|               sub_category|                           category_path|price|release_date|sales_rank|
+----------+----------------------------------------+-----------------------------------+--------------------+---------------------------+----------------------------------------+-----+------------+----------+
|0060219602|Hurray for Hattie Rabbit: Story and p...|Visit Amazon's Dick Gackenbach Page|               Books|eBook Readers & Accessories|Electronics > eBook Readers & Accesso...| null|        null|      null|
|0101635370|abcGoodefg&reg; 4GB USB 2.0 Mp3 Music...|                         Crazy Cart|     All Electronics|     Portable Audio & Video|Electronics > Portable

756061

## `prices`

Problemas encontrados no profiling e o que se faz:
- `prices.dateSeen` tem **várias datas na mesma célula**, separadas por vírgulas (até 198): cada data passa a uma linha (`explode`).
- As datas estão em **ISO 8601 UTC** (`2017-03-30T06:00:00Z`): passam a `DATE` (`date_seen`).
- `prices.condition` está escrito de várias formas (`New`, `new`, `Brand New`, e até descrições de produtos): ficam só os **novos**.
- `prices.currency` tem CAD, EUR, SGD e GBP em poucas linhas: ficam só os preços em **USD**.
- `prices.merchant` tem o mesmo retalhista escrito de formas diferentes (`Bestbuy.com` e `Best Buy`): é normalizado.
- `prices.isSale` não é fiável: a promoção real é `amount_min < amount_max` (`is_discount`), e o desconto é `discount_pct`.
- Valores passam de texto a `DOUBLE`/`BOOLEAN`, e os nomes das colunas passam a `snake_case`.

In [9]:
# create prices table in silver
spark.sql("DROP TABLE IF EXISTS silver_electronics.prices")

spark.sql(
    f"""
    CREATE EXTERNAL TABLE silver_electronics.prices (
        product_id STRING,
        name STRING,
        brand STRING,
        manufacturer_number STRING,
        categories STRING,
        asins STRING,
        merchant STRING,
        amount_min DOUBLE,
        amount_max DOUBLE,
        is_sale BOOLEAN,
        is_discount BOOLEAN,
        discount_pct DOUBLE,
        date_seen DATE
    )
    USING DELTA
    LOCATION '{SILVER}/prices/'
    """
)

DataFrame[]

In [10]:
# read prices (as mesmas opções do profiling: células com vírgulas e quebras de linha entre aspas)
prices_raw = spark.read.csv(f"{BRONZE}/DatafinitiElectronicsProductsPricingData.csv",
                            header=True, multiLine=True, escape='"')
prices_raw = prices_raw.toDF(*[c.replace('.', '_') for c in prices_raw.columns])

# o mesmo retalhista escrito de formas diferentes
m = F.lower(F.trim("prices_merchant"))
merchant = F.when(m.isin("bestbuy.com", "best buy"), "Best Buy") \
    .when(m == "walmart.com", "Walmart") \
    .when(m.isin("amazon.com", "amazon"), "Amazon") \
    .when(m == "bhphotovideo.com", "B&H Photo") \
    .when(m.isin("electronic_express", "electronic express"), "Electronic Express") \
    .when(m.isin("wwstereo", "world wide stereo"), "World Wide Stereo") \
    .otherwise(F.trim("prices_merchant"))

# apply filters and enhancements
prices_df = prices_raw \
    .withColumn("date_seen", F.explode(F.split("prices_dateSeen", ","))) \
    .withColumn("date_seen", F.to_date(F.to_timestamp(F.trim("date_seen")))) \
    .filter(F.col("prices_currency") == "USD") \
    .filter(F.lower(F.trim("prices_condition")).isin("new", "brand new")) \
    .withColumn("amount_min", F.col("prices_amountMin").cast("DOUBLE")) \
    .withColumn("amount_max", F.col("prices_amountMax").cast("DOUBLE")) \
    .filter((F.col("amount_min") > 0) & (F.col("amount_min") <= F.col("amount_max"))) \
    .withColumn("is_sale", F.upper("prices_isSale") == "TRUE") \
    .withColumn("is_discount", F.col("amount_min") < F.col("amount_max")) \
    .withColumn("discount_pct", F.round((F.col("amount_max") - F.col("amount_min")) / F.col("amount_max") * 100, 2)) \
    .withColumn("merchant", merchant) \
    .select(
        F.col("id").alias("product_id"),
        limpa_texto(F.col("name")).alias("name"),
        limpa_texto(F.col("brand")).alias("brand"),
        F.trim("manufacturerNumber").alias("manufacturer_number"),
        limpa_texto(F.col("categories")).alias("categories"),
        F.trim("asins").alias("asins"),
        "merchant", "amount_min", "amount_max", "is_sale", "is_discount", "discount_pct", "date_seen",
    ) \
    .drop_duplicates()   # o mesmo preço visto 2 vezes no mesmo dia conta uma vez

# write to delta table
prices_df \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER}/prices/")

# check the results in the table
spark.table("silver_electronics.prices").show(5, 30)
spark.table("silver_electronics.prices").count()

+--------------------+------------------------------+---------+-------------------+------------------------------+---------------------+---------+----------+----------+-------+-----------+------------+----------+
|          product_id|                          name|    brand|manufacturer_number|                    categories|                asins| merchant|amount_min|amount_max|is_sale|is_discount|discount_pct| date_seen|
+--------------------+------------------------------+---------+-------------------+------------------------------+---------------------+---------+----------+----------+-------+-----------+------------+----------+
|AVpiSS1A1cnluZ0-LyAY|Transcend - JetDrive 420 48...|Transcend|       TS480GJDM420|Computers,Hard Drives,Inter...|B00JQXT82I,B00QXMI1HI| Best Buy|    270.99|    270.99|   true|      false|         0.0|2014-09-28|
|AV13iAUYGV-KLJ3aka9M|Samsung - 50 Class (49.5" D...|  Samsung|      UN50M5300FXZA|LCD TVs,Samsung TVs,Electro...|           B0725FX4RF| Best Buy|  

33695

In [11]:
# confirmação: retalhistas depois da normalização
spark.table("silver_electronics.prices").groupBy("merchant").count().orderBy(F.desc("count")).show(10, False)

+------------------+-----+
|merchant          |count|
+------------------+-----+
|Best Buy          |15372|
|B&H Photo         |9279 |
|Walmart           |2086 |
|Beach Camera      |570  |
|null              |249  |
|buydig            |213  |
|Electronic Express|162  |
|Focus Camera      |161  |
|AMI Ventures Inc  |128  |
|echo-and-optics   |109  |
+------------------+-----+
only showing top 10 rows



## `prices_amazon_link`: join entre os dois datasets

Liga cada produto da Datafiniti ao(s) produto(s) da Amazon, para a Q1 (preço nos retalhistas vs avaliações na Amazon):
1. **`asin`**: a coluna `asins` da Datafiniti tem o código Amazon do produto (às vezes vários, separados por vírgulas).
2. **`model_brand`**: para os que não ligam pelo asin, mesma marca e o código do modelo (`manufacturer_number`, 4+ caracteres) aparece no título da Amazon.

A coluna `match_type` diz por qual das duas regras foi feita a ligação.

In [12]:
# create prices_amazon_link table in silver
spark.sql("DROP TABLE IF EXISTS silver_electronics.prices_amazon_link")

spark.sql(
    f"""
    CREATE EXTERNAL TABLE silver_electronics.prices_amazon_link (
        product_id STRING,
        asin STRING,
        match_type STRING
    )
    USING DELTA
    LOCATION '{SILVER}/prices_amazon_link/'
    """
)

DataFrame[]

In [13]:
# read data from the silver tables
prices_products = spark.table("silver_electronics.prices") \
    .select("product_id", "brand", "manufacturer_number", "asins").distinct()
amazon_df = spark.table("silver_electronics.amazon_products").select("asin", "brand", "title")

# 1) pelo asin
by_asin = prices_products \
    .withColumn("asin", F.explode(F.split("asins", ","))) \
    .withColumn("asin", F.trim("asin")) \
    .join(amazon_df.select("asin"), "asin") \
    .select("product_id", "asin") \
    .withColumn("match_type", F.lit("asin"))

# 2) pelo modelo + marca, só para os que não ligaram pelo asin
sem_ligacao = prices_products.join(by_asin, "product_id", "left_anti")
p, a = sem_ligacao.alias("p"), amazon_df.alias("a")
by_model = p.join(
        a,
        (F.lower(F.col("p.brand")) == F.lower(F.col("a.brand")))
        & (F.length("p.manufacturer_number") >= 4)
        & F.lower(F.col("a.title")).contains(F.lower(F.col("p.manufacturer_number"))),
    ) \
    .select("p.product_id", "a.asin") \
    .withColumn("match_type", F.lit("model_brand"))

link_df = by_asin.union(by_model).drop_duplicates(["product_id", "asin"])

# write to delta table
link_df \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{SILVER}/prices_amazon_link/")

# check the results in the table
spark.sql("""
    SELECT match_type, COUNT(DISTINCT product_id) AS produtos, COUNT(*) AS pares
    FROM silver_electronics.prices_amazon_link
    GROUP BY match_type
""").show()
print("produtos Datafiniti:", prices_products.select("product_id").distinct().count())

+-----------+--------+-----+
| match_type|produtos|pares|
+-----------+--------+-----+
|       asin|     501|  525|
|model_brand|      39|   50|
+-----------+--------+-----+

produtos Datafiniti: 1240


## Verificação final

In [14]:
spark.sql("USE silver_electronics")
spark.sql("SHOW TABLES").show()

for t in ["amazon_ratings", "amazon_products", "prices", "prices_amazon_link"]:
    print(t, spark.table(f"silver_electronics.{t}").count())

+------------------+------------------+-----------+
|         namespace|         tableName|isTemporary|
+------------------+------------------+-----------+
|silver_electronics|   amazon_products|      false|
|silver_electronics|    amazon_ratings|      false|
|silver_electronics|            prices|      false|
|silver_electronics|prices_amazon_link|      false|
+------------------+------------------+-----------+

amazon_ratings 20568091
amazon_products 756061
prices 33695
prices_amazon_link 575


In [15]:
# tipos finais (DATE, INT, DOUBLE, BOOLEAN em vez de texto)
spark.table("silver_electronics.prices").printSchema()
spark.sql("DESCRIBE HISTORY silver_electronics.prices").select("version", "timestamp", "operation").show()

root
 |-- product_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- manufacturer_number: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- asins: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- amount_min: double (nullable = true)
 |-- amount_max: double (nullable = true)
 |-- is_sale: boolean (nullable = true)
 |-- is_discount: boolean (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- date_seen: date (nullable = true)

+-------+--------------------+------------+
|version|           timestamp|   operation|
+-------+--------------------+------------+
|      3|2026-11-12 09:47:...|       WRITE|
|      2|2026-11-11 23:03:...|       WRITE|
|      1|2026-11-11 23:00:...|       WRITE|
|      0|2026-11-11 23:00:...|CREATE TABLE|
+-------+--------------------+------------+



In [16]:
# Fecha a sessão spark
spark.stop()